# Day 3 · 连接器：模态对齐的关键那一层

**配套讲义**: [`days/day-03.md`](../days/day-03.md) ｜ **需要 GPU（云机器）**

实现三种连接器（单层 MLP / 两层 MLP+GELU / Perceiver Resampler），对比它们的参数量和输出 token 数，并回答「为什么 LLaVA 用最笨的 MLP 反而效果好」。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w1.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 三种连接器并排看

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.minivlm.connector"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 2. 手算 MLP 连接器的参数量

**算一遍你就不会忘了。** 视觉侧 D=1152，LLM 侧 D=2048。

In [ ]:
D_v, D_l, D_h = 1152, 2048, 2048   # D_h 是 2-layer 的隐层宽

mlp1  = D_v * D_l + D_l
mlp2  = (D_v * D_h + D_h) + (D_h * D_l + D_l)
print(f"单层 MLP  : {mlp1/1e6:.2f} M")
print(f"两层 MLP  : {mlp2/1e6:.2f} M")

# 相对 3B 的 LLM 主干
print(f"\n占 3B 模型的比例: {mlp2/3e9:.3%}")
print("→ 连接器确实很小，但它决定了视觉信息『以什么形式』进入 LLM")

## 3. token 数 vs 压缩率

Perceiver 的 K 是**唯一的压缩旋钮**。把它拉大拉小，看列表怎么变。

In [ ]:
N = 1024
print(f"{'K':>6} {'输出token':>10} {'压缩率':>8}  {'适用场景':<28}")
print("-" * 60)
for K, note in [(16, "丢信息过多，细节问题必错"),
                (64, "Flamingo 的默认，多图/视频"),
                (256, "较保守，接近不压缩"),
                (1024, "= 不压缩，那不如直接用 MLP")]:
    print(f"{K:>6} {K:>10} {N/K:>7.0f}×  {note:<28}")

print("\n→ 注意 K=1024 那一行：既然不压缩，为什么要多一层注意力？"
      "\n  这就是 LLaVA 选 MLP 的立场 —— 压缩交给 LLM 自己去做")

## 4. 打卡

In [ ]:
print("""今日打卡
─────────────────────────────────────────
[学到] 连接器的作用是 ______；LLaVA 选 MLP 的原因是 ______
[产出] src/minivlm/connector.py 三种实现 + 对比表
[卡住] ______
─────────────────────────────────────────""")

## 验收清单

- [ ] 能画出三种连接器的结构图，并说清 query 的数量各自是多少
- [ ] 能回答「LLaVA 为什么用 MLP 反而好」——**答案和 LLM 的强度有关**
- [ ] 能说出 Perceiver 的 K 调大调小分别会怎样（K 太小丢信息，K 太大失去压缩意义）
- [ ] 知道连接器参数量占比很小，但决定信息以什么形式进 LLM

**卡住了？** 回看 [`days/day-03.md`](../days/day-03.md) 第五节「容易踩的坑」。

> **明天**：`days/day-04.md` —— 精读 Qwen2.5-VL：M-RoPE 和 2×2 patch merging